In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/train_valid_sequences_quelevel.csv
/kaggle/input/datasets/nht123456789/qikt-f1234-s42-pykt-scripts/DKT_A_f0_preds.parquet
/kaggle/input/datasets/nht123456789/qikt-f1234-s42-pykt-scripts/fix_pykt_raw.py
/kaggle/input/datasets/nht123456789/qikt-f1234-s42-pykt-scripts/

In [2]:
%%bash
# Một ô Kaggle cho các lần chạy pyKT (bản 26/10): huấn luyện -> predict -> registry.
# dkt, akt, simplekt...: predict_sharded.py (mức KC); qikt, iekt: predict_quelevel.py.
# Mỗi fold một thư mục run, có thêm model.ckpt và 3 log (train, predict, RAM) cho E1/E2.
set -e
cd /kaggle/working
[ -d pykt-toolkit ] || git clone -q https://github.com/pykt-team/pykt-toolkit.git
cd pykt-toolkit && git checkout -q eec087e   # ghim commit
T=/kaggle/input/datasets/nht123456789/qikt-f1234-s42-pykt-scripts
bash $T/pykt_qlevel_device_patch.sh   
pip install -q -e . && pip install -q einops entmax
X=/kaggle/input/datasets/nhatanhadr/xes3g5m-extended    # một Dataset cho mọi notebook
REF=$T/DKT_A_f0_preds.parquet
mkdir -p data/peiyou
for f in train_valid_sequences test_question_window_sequences \
         train_valid_sequences_quelevel test_window_sequences_quelevel; do
  ln -sf $X/$f.csv data/peiyou/$f.csv             # liên kết, không chép: tiết kiệm đĩa
done
cp $T/*.py examples/ && cd examples && mkdir -p logs
M=qikt; S=42; FOLDS="1 2 3 4"                    # SỬA: mô hình, seed, các fold của phiên này
EXTRA=""; PRED=predict_sharded.py; PARGS=""
case $M in
  qikt) EXTRA="--output_mode=an_irt"; PRED=predict_quelevel.py; PARGS="--seed=$S" ;;
  iekt) PRED=predict_quelevel.py; PARGS="--seed=$S" ;;
esac
trap 'kill $RAM 2>/dev/null || true' EXIT       # dừng vòng ghi RAM cả khi lỗi
for F in $FOLDS; do
  L=logs/${M}_f${F}_s${S}
  ( while true; do echo "$(date +%T) $(free -g | awk '/Mem/ {print $3"/"$2}') GB"
      sleep 15; done ) > ${L}_ram.log &
  RAM=$!
  python -u wandb_${M}_train.py --dataset_name=peiyou --fold=$F --seed=$S $EXTRA \
    --use_wandb=0 --add_uuid=0 --save_dir=sm_${M}_f${F} > ${L}_train.log 2>&1
  CKPT=$(dirname $(find sm_${M}_f${F} -name config.json | head -n 1))
  
  # --- ĐO THỜI GIAN PREDICT ---
  T0=$SECONDS
  python -u $PRED --save_dir=$CKPT --n_shards=8 --bz=64 $PARGS > ${L}_predict.log 2>&1
  echo "predict_s=$((SECONDS-T0))" >> ${L}_predict.log
  # -----------------------------
  
  kill $RAM
  python pykt_to_registry.py --save_dir=$CKPT --train_log=${L}_train.log \
    --test_csv=$X/test.csv --pykt_commit=eec087e --runs_root=/kaggle/working/runs \
    --ref_preds=$REF --logs ${L}_predict.log ${L}_ram.log
  tail -3 ${L}_predict.log
  echo "RAM cao nhất fold $F: $(awk '{print $2}' ${L}_ram.log | sort -n | tail -1) GB"
done
cat /kaggle/working/runs/registry.csv

đã vá qikt.py, iekt.py, que_base_model.py
QIKT_pyKT_f1_s42_20261103-0954 {'auc': 0.8208, 'acc': 0.8296, 'rmse': 0.3494, 'brier': 0.1221, 'ece': 0.004, 'auprc_wrong': 0.5755, 'f1_wrong': 0.4492}
window_testauc = 0.820779
window_testacc = 0.829574
predict_s=1204
RAM cao nhất fold 1: 5/31 GB
QIKT_pyKT_f2_s42_20261103-1049 {'auc': 0.8207, 'acc': 0.8293, 'rmse': 0.3495, 'brier': 0.1221, 'ece': 0.0034, 'auprc_wrong': 0.5752, 'f1_wrong': 0.4402}
window_testauc = 0.820680
window_testacc = 0.829253
predict_s=1185
RAM cao nhất fold 2: 5/31 GB
QIKT_pyKT_f3_s42_20261103-1143 {'auc': 0.8205, 'acc': 0.8295, 'rmse': 0.3495, 'brier': 0.1222, 'ece': 0.006, 'auprc_wrong': 0.5753, 'f1_wrong': 0.4382}
window_testauc = 0.820472
window_testacc = 0.829456
predict_s=1187
RAM cao nhất fold 3: 5/31 GB
QIKT_pyKT_f4_s42_20261103-1238 {'auc': 0.8205, 'acc': 0.8293, 'rmse': 0.3502, 'brier': 0.1226, 'ece': 0.0207, 'auprc_wrong': 0.5755, 'f1_wrong': 0.4654}
window_testauc = 0.820516
window_testacc = 0.829261
predict_